In [ ]:
from dotenv import load_dotenv

load_dotenv()  # reads .env file from the current directory

In [ ]:
from pathlib import Path

PATH_DATA = Path.cwd().parent / ".data"
PATH_GPT2_124M_WEIGHTS = PATH_DATA / "model_weights" / "gpt2" / "124M"
PATH_GPT2_124M_WEIGHT_SETTINGS = PATH_GPT2_124M_WEIGHTS / "settings.pickle.gz"
PATH_GPT2_124M_WEIGHTS_PARAMETERS = PATH_GPT2_124M_WEIGHTS / "parameters.pickle.gz"

DEFAULT_CONTEXT_LENGTH = 1024
DEFAULT_STRIDE = 1

In [3]:
import tiktoken
from tgedr_languagemodels.configuration import GPT2_MODEL_CONFIGS, BaseModelConfig
from tgedr_languagemodels.gpt2.model import GPT2Model
from tgedr_languagemodels.utils.model_weights import load_weights_into_gpt

model_name = "gpt2-small (124M)"
tokenizer = tiktoken.get_encoding("gpt2")

config: BaseModelConfig = BaseModelConfig(
    vocabulary_size=tokenizer.n_vocab,
    context_length=DEFAULT_CONTEXT_LENGTH,
    embeddings_dimension=GPT2_MODEL_CONFIGS[model_name]["emb_dim"],
    n_heads=GPT2_MODEL_CONFIGS[model_name]["n_heads"],
    n_layers=GPT2_MODEL_CONFIGS[model_name]["n_layers"],
    drop_rate=0.1,
    qkv_bias=True,
    stride=DEFAULT_STRIDE
)

model = GPT2Model(config)
model.eval()

GPT2Model(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_projection): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNormalization()
      (norm2): LayerNormalization()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_que

In [4]:
from tgedr_languagemodels.utils.utils_llm import load_pickle_compressed

pre_trained_params = load_pickle_compressed(PATH_GPT2_124M_WEIGHTS_PARAMETERS)
load_weights_into_gpt(model, pre_trained_params)

In [5]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model.to(device)

Using device: cpu


GPT2Model(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_projection): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNormalization()
      (norm2): LayerNormalization()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_que

In [6]:
from tgedr_languagemodels.inference import generate
from tgedr_languagemodels.utils.utils_llm import text_to_token_ids, token_ids_to_text

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=25,
    context_size=config.context_length,
    top_k=50,
    temperature=1.5,
)
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you into higher activity where you feel you might lose any part that isn't of greater importance that you can just rest your game.
